# 🔍 Inspección del Modelo — Análisis de Riesgos CDMX

Este notebook carga el modelo entrenado (`.joblib`), extrae su información
(tipo, hiperparámetros, features, importancia, métricas) y genera reportes
en **HTML**, **JSON** y **Markdown**.

Los reportes se guardan en `outputs/modelo/modelo_inspector/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from src.config import setup_logging, ensure_output_dirs, MODELO_DIR, MODELO_INSPECTOR_DIR
setup_logging()
ensure_output_dirs()

print(f"Directorio del modelo : {MODELO_DIR}")
print(f"Directorio de reportes: {MODELO_INSPECTOR_DIR}")

## 1. Cargar el modelo

In [ ]:
import joblib
from src.model_inspector import load_and_inspect

# Cargar e inspeccionar
info = load_and_inspect()

print(f"Tipo de modelo: {info['model_type']}")
print(f"Num. features : {info['n_features']}")
if info.get('sub_models'):
    print(f"Sub-modelos   : {info['n_sub_models']}")
    for sub in info['sub_models']:
        print(f"  → {sub['type']}")

## 2. Hiperparámetros de sub-modelos

In [ ]:
import pandas as pd

if info.get('sub_models'):
    for sub in info['sub_models']:
        print(f"
=== {sub['type']} ===")
        params_df = pd.DataFrame(
            list(sub['hyperparameters'].items()),
            columns=['Parámetro', 'Valor']
        )
        display(params_df)
elif info.get('hyperparameters'):
    params_df = pd.DataFrame(
        list(info['hyperparameters'].items()),
        columns=['Parámetro', 'Valor']
    )
    display(params_df)

## 3. Importancia de Features

In [ ]:
import matplotlib.pyplot as plt

if info.get('feature_importances'):
    imp = info['feature_importances']
    features = list(imp.keys())
    values = list(imp.values())
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.RdYlGn_r([v / max(values) for v in values])
    ax.barh(features[::-1], values[::-1], color=colors[::-1])
    ax.set_xlabel('Importancia')
    ax.set_title('Importancia de Features — Modelo Final', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No se encontró información de importancia de features.")

## 4. Métricas de evaluación

In [ ]:
if info.get('metrics'):
    metrics_df = pd.DataFrame(
        [(k, f"{v:.4f}") for k, v in info['metrics'].items()],
        columns=['Métrica', 'Valor']
    )
    display(metrics_df)
else:
    print("No se encontraron métricas guardadas.")

## 5. Generar reportes

In [ ]:
from src.model_inspector import generate_all_reports

paths = generate_all_reports()

print("
📄 Reportes generados:")
for fmt, p in paths.items():
    print(f"  [{fmt.upper():>8}] → {p}")

## 6. Vista previa del reporte HTML

In [ ]:
from IPython.display import HTML, display
from pathlib import Path

html_path = paths.get('html')
if html_path and Path(html_path).exists():
    html_content = Path(html_path).read_text(encoding='utf-8')
    display(HTML(html_content))
else:
    print("Reporte HTML no disponible.")

---
✅ **Inspección completada.** Los reportes están disponibles en:
- 
- 
-